<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z319_AutoGluon_Racha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon + Test de Racha

## ¿Cuál es la idea?

AutoGluon es poderoso pero ciego: recibe las 780 series y les aplica los mismos modelos a todas, sin distinguir cuáles tienen estructura temporal real y cuáles son esencialmente ruido.

El **Test de Racha** nos permite hacer esa distinción *antes* de entrenar:

- **Serie con estructura** (p-value < α): AutoGluon puede aprender algo útil → la dejamos entrar al entrenamiento normal
- **Serie sin estructura / ruido** (p-value ≥ α): AutoGluon va a sobreajustar sobre ruido → la reemplazamos directamente por el **promedio de los últimos 12 meses** y no la mandamos a AutoGluon

El resultado es un submit **híbrido**: AutoGluon para las series con señal, promedio simple para las series ruidosas.

### ¿Por qué esto puede mejorar el score?

AutoGluon incluye modelos que buscan patrones complejos (DeepAR, TFT). En una serie ruidosa, esos modelos van a encontrar patrones que no existen y van a predecir peor que un simple promedio. Filtrar esas series antes del entrenamiento reduce el ruido en la señal de validación y deja que AutoGluon se concentre donde realmente puede aportar.

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  AutoGluon + Test de Racha

## 1.1 Init Experimento

In [ ]:
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
from datetime import datetime
from scipy.stats import runstest_1samp
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

import warnings
warnings.filterwarnings('ignore')

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida.
<br> `alpha_runs`: nivel de significancia del Test de Racha. Con 0.05 es conservador (solo clasifica como 'con estructura' las series claramente no aleatorias). Probar también con 0.10 para ser mas permisivo.

In [ ]:
PARAM = {
  'experimento': 'AutoGluon_Racha-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  'alpha_runs': 0.05   # umbral del test: bajar = mas series van a AutoGluon
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.2 Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [ ]:
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

In [ ]:
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner")
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [ ]:
# paso de periodo a timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

## 1.3 Test de Racha por producto

El **Test de Racha (Wald-Wolfowitz)** evalúa si la secuencia de valores de una serie es aleatoria o tiene estructura temporal.

Mecanismo: convierte la serie en una secuencia binaria (`+` si el valor está por encima de la mediana, `-` si está por debajo) y cuenta las **rachas** — tramos consecutivos del mismo signo.

- **Pocas rachas** → los valores altos se agrupan y los bajos se agrupan → hay **autocorrelación positiva** (tendencia, estacionalidad)
- **Muchas rachas** → los valores alternan constantemente → **autocorrelación negativa**
- **Cantidad esperada bajo H₀** → la serie es ruido blanco

**Resultado práctico**:
- `p-value < alpha` → rechazamos aleatoriedad → la serie tiene estructura → **la mandamos a AutoGluon**
- `p-value ≥ alpha` → no hay evidencia de estructura → **promedio 12m directamente**, sin gastar tiempo de AutoGluon en ella

In [ ]:
productos = tb_apredecir["product_id"].to_list()

runs_resultados = []

for producto in productos:
    serie = (
        tb_ventas
        .filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"]
        .to_numpy()
        .astype(float)
    )

    try:
        _, pvalue = runstest_1samp(serie, cutoff='median')
        tiene_estructura = bool(pvalue < PARAM['alpha_runs'])
    except Exception:
        pvalue = np.nan
        tiene_estructura = False

    # fallback: promedio ultimos 12 meses
    promedio_12m = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    runs_resultados.append({
        'product_id': producto,
        'pvalue_runs': round(float(pvalue), 4) if not np.isnan(pvalue) else None,
        'tiene_estructura': tiene_estructura,
        'promedio_12m': promedio_12m
    })

tb_runs = pl.DataFrame(runs_resultados)
display(tb_runs)

In [ ]:
# resumen: ¿cuantos van a AutoGluon y cuantos van directo a promedio?
n_total      = len(productos)
n_estructura = tb_runs["tiene_estructura"].sum()
n_ruido      = n_total - n_estructura

print(f"Total productos        : {n_total}")
print(f"Con estructura (→ AG)  : {n_estructura}  ({100*n_estructura/n_total:.1f}%)")
print(f"Sin estructura (→ prom): {n_ruido}  ({100*n_ruido/n_total:.1f}%)")

## 1.4 Entrenamiento AutoGluon — solo series con estructura

Le pasamos a AutoGluon únicamente las series que el Test de Racha identificó como no aleatorias. Las series ruidosas ya tienen su predicción asignada (promedio 12m) y no participan del entrenamiento.

Ventajas de este filtro:
- AutoGluon entrena y valida sobre series donde hay señal real — su métrica interna de validación es más honesta
- Se reduce el tiempo de entrenamiento proporcionalmente a las series filtradas
- Los modelos globales (DeepAR, TFT) no se contaminan con ruido de series inútiles

In [ ]:
# productos que van a AutoGluon
productos_con_estructura = tb_runs.filter(
    pl.col("tiene_estructura") == True
)["product_id"].to_list()

# filtro tb_ventas a solo esos productos
tb_ventas_ag = tb_ventas.filter(
    pl.col("product_id").is_in(productos_con_estructura)
)

print(f"Filas en tb_ventas original : {tb_ventas.height}")
print(f"Filas que van a AutoGluon   : {tb_ventas_ag.height}")

In [ ]:
# paso a formato TimeSeriesDataFrame con solo las series estructuradas
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas_ag.to_pandas(),
  timestamp_column='timestamp',
  id_column='product_id'
)

In [ ]:
# Entrenamiento AutoGluon
# https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html
global_eval_metric = 'RMSE'

modelo = TimeSeriesPredictor(
  prediction_length= 2,
  target= 'tn',
  freq= 'MS',
  eval_metric= global_eval_metric
)

modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600,
  presets= "best_quality",
  random_seed= PARAM['semilla_primigenia']
)

## 1.5 Prediccion AutoGluon + ensamble con promedio 12m

AutoGluon predice las series con estructura. Para las series ruidosas usamos el promedio 12m calculado antes del entrenamiento.

El resultado final cubre los 780 productos.

In [ ]:
# prediccion AutoGluon para las series con estructura
tb_forecast = modelo.predict(ts_data, random_seed=PARAM['semilla_primigenia'])
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

# me quedo con febrero 2020
tb_ag_final = (
    tb_forecast
    .filter(pl.col("timestamp") == datetime(2020, 2, 1))
    .select(["item_id", "mean"])
    .rename({"item_id": "product_id", "mean": "tn"})
)

display(tb_ag_final)

In [ ]:
# predicciones de los productos ruidosos: promedio 12m
tb_ruido_final = (
    tb_runs
    .filter(pl.col("tiene_estructura") == False)
    .select(["product_id", "promedio_12m"])
    .rename({"promedio_12m": "tn"})
)

print(f"Productos de AutoGluon : {tb_ag_final.height}")
print(f"Productos de promedio  : {tb_ruido_final.height}")

In [ ]:
# uno ambas partes
tb_final = pl.concat([tb_ag_final, tb_ruido_final]).sort("product_id")

# valores negativos a cero
tb_final = tb_final.with_columns(
    pl.when(pl.col("tn") < 0).then(0.0).otherwise(pl.col("tn")).alias("tn")
)

display(tb_final)
print(f"Total productos: {tb_final.height}")
print(f"Nulls          : {tb_final['tn'].is_null().sum()}")

## 1.6 Submit a Kaggle

In [ ]:
archivo = "AutoGluon_Racha_" + global_eval_metric + ".csv"
mensaje = "AutoGluon + Test de Racha alpha=" + str(PARAM['alpha_runs']) + " " + global_eval_metric

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)